# CS224N L01 Code Capsule: Negative Sampling Loss vs Full Softmax

**Waypoint**: WP05 — 负采样 (Skip-gram Negative Sampling)

**Concept**: 用 logistic 二分类替代 softmax 全词表归一化，解决计算效率瓶颈

**Official Anchor**: Notes §3.5 (Eq.14-15, SGNS objective); R02 paper Section 3

**Key equations**:
- Eq.14: $P(c|o) = \frac{\exp(u_c^\top v_o)}{\sum_w \exp(u_w^\top v_o)}$
- Eq.15: $J_{NS} = -\log\sigma(u_o^\top v_c) - \sum_{\ell=1}^{k}\log\sigma(-u_\ell^\top v_c)$

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

np.random.seed(42)

VOCAB_SIZE = 10000
DIM = 50

U = np.random.randn(VOCAB_SIZE, DIM) * 0.1
V = np.random.randn(VOCAB_SIZE, DIM) * 0.1

center_idx, context_idx = 42, 100
v_c = V[center_idx]
u_o = U[context_idx]

print(f"|V| = {VOCAB_SIZE}, dim = {DIM}")
print(f"u_o^T v_c = {np.dot(u_o, v_c):.4f}")

## Part 1: Full Softmax Loss (Eq.14)

The partition function Z sums over ALL |V| words — the bottleneck!

In [ ]:
scores = U @ v_c
scores_shifted = scores - np.max(scores)
exp_scores = np.exp(scores_shifted)
partition_function = np.sum(exp_scores)
softmax_probs = exp_scores / partition_function
softmax_loss = -np.log(softmax_probs[context_idx] + 1e-10)

print(f"Partition Z = {partition_function:.6f}")
print(f"P(context|center) = {softmax_probs[context_idx]:.8f}")
print(f"Softmax loss = {softmax_loss:.6f}")
print(f"Required {VOCAB_SIZE} exp() evaluations!")

## Part 2: Negative Sampling Loss (Eq.15)

Only k+1 dot products instead of |V|!

In [ ]:
def sigmoid(x):
    return np.where(x >= 0, 1.0/(1.0+np.exp(-x)), np.exp(x)/(1.0+np.exp(x)))

def ns_loss(v_c, u_o, U_neg, k):
    pos = -np.log(sigmoid(np.dot(u_o, v_c)) + 1e-10)
    neg = -np.sum(np.log(sigmoid(-(U_neg @ v_c)) + 1e-10))
    return pos + neg

k_values = [1, 5, 10, 25]
print(f"{'k':>4} | {'NS Loss':>10} | {'Softmax':>10} | {'Diff':>8}")
print("-"*40)
for k in k_values:
    neg_idx = np.random.choice(VOCAB_SIZE, size=k, replace=False)
    while context_idx in neg_idx:
        neg_idx = np.random.choice(VOCAB_SIZE, size=k, replace=False)
    l = ns_loss(v_c, u_o, U[neg_idx], k)
    print(f"{k:>4} | {l:>10.6f} | {softmax_loss:>10.6f} | {l-softmax_loss:>+8.4f}")

print("\nNS != softmax! NS is binary classification, not probability matching.")

## Part 3: Sigmoid & Gradients

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.linspace(-5, 5, 200)
axes[0].plot(x, sigmoid(x), 'b-', lw=2, label='σ(x) — positive pair')
axes[0].plot(x, sigmoid(-x), 'r-', lw=2, label='σ(-x) — negative pair')
axes[0].set_xlabel('Score (u^T v)'); axes[0].set_ylabel('σ(score)')
axes[0].set_title('NS = Two Logistic Classifications'); axes[0].legend(); axes[0].grid(alpha=0.3)

sr = np.linspace(-4, 4, 100)
axes[1].plot(sr, 1-sigmoid(sr), 'b-', lw=2, label='Pos: 1-σ(u_o^T v_c)')
axes[1].plot(sr, 1-sigmoid(-sr), 'r-', lw=2, label='Neg: 1-σ(-u_ℓ^T v_c)')
axes[1].set_xlabel('Score'); axes[1].set_ylabel('Gradient magnitude')
axes[1].set_title('When Does Each Sample Push Hard?'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/negative-sampling-loss-sigmoid-gradients.png', dpi=150, bbox_inches='tight')
plt.show()

## Part 4: Effect of k on Variance

In [ ]:
k_range = [1, 2, 5, 10, 20, 50]
means, stds = [], []
for k in k_range:
    losses = [ns_loss(v_c, u_o, U[np.random.choice(VOCAB_SIZE, k, replace=False)], k) for _ in range(100)]
    means.append(np.mean(losses)); stds.append(np.std(losses))
    print(f"k={k:>2}: mean={np.mean(losses):.4f}, std={np.std(losses):.4f}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.errorbar(k_range, means, yerr=stds, fmt='o-', capsize=4, color='#2196F3', lw=2)
ax.axhline(softmax_loss, color='r', ls='--', lw=2, label=f'Softmax={softmax_loss:.4f}')
ax.set_xlabel('k'); ax.set_ylabel('Loss'); ax.set_title('NS Loss vs k'); ax.set_xscale('log'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/negative-sampling-loss-comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Part 5: Efficiency — Gradient Computation

In [ ]:
vocab_sizes = [1000, 10000, 50000, 100000]
t_sm, t_ns = {}, {}
for vs in vocab_sizes:
    Ut = np.random.randn(vs, DIM) * 0.1
    t0 = time.perf_counter()
    for _ in range(50):
        s = Ut @ v_c; s -= s.max(); e = np.exp(s); p = e/e.sum()
        g = p[:,None] * v_c
    t_sm[vs] = (time.perf_counter()-t0)/50
    
    t0 = time.perf_counter()
    for _ in range(50):
        ni = np.random.choice(vs, 5, replace=False); Un = Ut[ni]
        ps = np.dot(Ut[context_idx%vs], v_c); ns_ = Un @ v_c
        sp = sigmoid(ps); sn = sigmoid(-ns_)
        g = (sp-1)*Ut[context_idx%vs]
        for i in range(5): g += (1-sn[i])*Un[i]
    t_ns[vs] = (time.perf_counter()-t0)/50

print(f"{'|V|':>8} | {'Softmax':>10} | {'NS(k=5)':>10} | {'Speedup':>8}")
for vs in vocab_sizes:
    print(f"{vs:>8} | {t_sm[vs]*1000:>8.3f}ms | {t_ns[vs]*1000:>8.3f}ms | {t_sm[vs]/t_ns[vs]:>6.1f}x")

## Summary

| Aspect | Softmax | Negative Sampling |
|--------|---------|-------------------|
| Dot products | |V| | k+1 |
| Gradient touches | All |V| vectors | k+1 vectors |
| Complexity | O(|V|·d) | O(k·d) |

**Key**: NS is NOT an approximation of softmax — it's a different objective (binary classification) that implicitly learns similar representations.

**Glossary**: [[Lectures/L01-word-vectors/00-concept-glossary#softmax|softmax]], [[Lectures/L01-word-vectors/00-concept-glossary#skip-gram|skip-gram]]